In [1]:
from pathlib import Path 
import pinocchio as pin

In [ ]:
pinocchio_model_dir = Path(__file__).parent
model_path = Path((pinocchio_model_dir / "example-robot-data/robots")) 
mesh_dir = pinocchio_model_dir 
urdf_model_path = model_path / "romeo_description/urdf/romeo_small.urdf" 

# 1. Load model and collision geometries

In [ ]:
model = pin.buildModelFromUrdf(urdf_model_path, pin.JointModelFreeFlyer())
geom_model = pin.buildGeomFromUrdf(
    model, urdf_model_path, pin.GeometryType.COLLISION, mesh_dir
)

# 2. Add collisition pairs

In [ ]:
geom_model.addAllCollisionPairs()
print("num collision pairs - initial:", len(geom_model.collisionPairs))

# 3. Remove collision pairs listed in the SRDF file

In [ ]:
srdf_filename = "romeo.srdf"
srdf_model_path = model_path / "romeo_description/srdf" / srdf_filename

pin.removeCollisionPairs(model, geom_model, srdf_model_path)
print(
    "num collision pairs - after removing useless collision pairs:",
    len(geom_model.collisionPairs),
)

# 4. Load reference configuration

In [ ]:
pin.loadReferenceConfigurations(model, srdf_model_path)

q = model.referenceConfigurations["half_sitting"]

# 5. Compute all the collisions

In [ ]:
data = model.createData()
geom_data = pin.GeometryData(geom_model)

In [ ]:
pin.computeCollisions(model, data, geom_model, geom_data, q, False)

In [ ]:
# Print the status of collision for all collision pairs
for k in range(len(geom_model.collisionPairs)):
    cr = geom_data.collisionResults[k]
    cp = geom_model.collisionPairs[k]
    print(
        "collision pair:",
        cp.first,
        ",",
        cp.second,
        "- collision:",
        "Yes" if cr.isCollision() else "No",
    )

In [ ]:
# Compute for a single pair of collision
pin.updateGeometryPlacements(model, data, geom_model, geom_data, q)
pin.computeCollision(geom_model, geom_data, 0)